# Название проекта

| Поле | Значение |
|---|---|
| Автор | Имя Фамилия |
| Итерация | 1 / 2 / 3 |
| Направление | Классификация / Регрессия / Кластеризация |
| Тип задачи | Например: бинарная классификация |
| Датасет | Название и ссылка |
| Метод(ы) | Например: логистическая регрессия, kNN |
| Дата | дд.мм.гггг |

**Цель проекта одним предложением:** …

> **Как пользоваться шаблоном.** Заполняйте разделы по порядку. Блоки «Подсказка» и комментарии `TODO` удалите после заполнения. Если раздел к вашей задаче не относится (например, разбиение на train/test для кластеризации), оставьте заголовок и напишите одну строку, почему он пропущен.

**Содержание:** 1. Теория · 2. Постановка задачи и данные · 3. Окружение · 4. Загрузка данных · 5. Первичный анализ (EDA) · 6. Предобработка · 7. Базовая модель · 8. Основная модель · 9. Оценка результатов · 10. Выводы · 11. Абзац для отчёта · 12. Рефлексия · 13. Чек-лист сдачи · 14. Источники

## 1. Теория

> **Подсказка.** 150–300 слов своими словами, без копирования. Не больше одной формулы или схемы. Минимум два источника в разделе 1.6.

### 1.1 Что это за задача
Что такое классификация / регрессия / кластеризация, чем она отличается от двух других направлений, какой у неё тип (бинарная, мультиклассовая, multilabel; простая, множественная, полиномиальная; K-Means, иерархическая, DBSCAN).

### 1.2 Как работает метод
Идея метода простыми словами. Что он делает с данными на этапе обучения и как получает ответ для нового объекта.

### 1.3 Ключевые параметры
Какие гиперпараметры у метода (например, `k` в kNN, `alpha` в Ridge, `eps` в DBSCAN) и на что они влияют.

### 1.4 Когда применять, плюсы и минусы
Для каких данных подходит, где ошибается, что чувствительно к масштабу, выбросам и дисбалансу.

### 1.5 Метрики качества
Какие метрики будем использовать и почему именно они (а не, например, accuracy при дисбалансе классов).

### 1.6 Материалы для чтения

| Источник | О чём | Прочитано |
|---|---|---|
| ссылка | кратко | да / нет |

### 1.7 Вопросы для самопроверки
1. Вопрос → ответ своими словами
2. Вопрос → ответ своими словами
3. Вопрос → ответ своими словами

## 2. Постановка задачи и данные

### 2.1 Задача
Что предсказываем или группируем, что является входом и выходом, кому и зачем это полезно.

### 2.2 Источник данных
Название, ссылка, количество строк и столбцов, период сбора, лицензия (если указана).

### 2.3 Описание столбцов

| Столбец | Тип | Смысл | Примечание |
|---|---|---|---|
| … | число / категория / текст | … | … |

### 2.4 Целевая переменная
Для классификации и регрессии: название столбца и что он означает. Для кластеризации: «целевой переменной нет».

## 3. Окружение

In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

warnings.filterwarnings('ignore')

RANDOM_STATE = 42          # единое значение для воспроизводимости
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

print('pandas', pd.__version__, '| scikit-learn', sklearn.__version__)

## 4. Загрузка данных

In [ ]:
# TODO: укажите путь к своему файлу (Kaggle Notebook или локальная папка data/)
CANDIDATES = [
    '/kaggle/input/НАЗВАНИЕ-ДАТАСЕТА/файл.csv',
    'data/файл.csv',
]
path = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert path, 'Файл не найден: проверьте пути в CANDIDATES'

df = pd.read_csv(path)
print('Размер таблицы:', df.shape)

## 5. Первичный анализ данных (EDA)

### 5.1 Первый взгляд

In [ ]:
display(df.head())
df.info()
display(df.describe(include='all').T)

### 5.2 Пропуски и дубликаты

In [ ]:
missing = df.isna().sum().to_frame('пропусков')
missing['доля'] = (missing['пропусков'] / len(df)).round(3)
display(missing[missing['пропусков'] > 0])
print('Дубликатов строк:', df.duplicated().sum())
# Подсказка: проверьте «скрытые пропуски», когда 0 или '?' означает «не измерено» (например, в Pima).

### 5.3 Целевая переменная
Для классификации: сколько классов и сбалансированы ли они. Для регрессии: форма распределения. Для кластеризации: пропустите.

In [ ]:
TARGET = 'target'   # TODO: имя целевого столбца; для кластеризации оставьте None

if TARGET:
    if df[TARGET].nunique() <= 20:
        display(df[TARGET].value_counts(normalize=True).rename('доля'))
        df[TARGET].value_counts().plot(kind='bar', title='Распределение классов')
    else:
        df[TARGET].plot(kind='hist', bins=30, title='Распределение целевой переменной')
    plt.show()

### 5.4 Распределения признаков

In [ ]:
df.select_dtypes('number').hist(figsize=(12, 8), bins=30)
plt.tight_layout()
plt.show()

### 5.5 Связи между признаками

In [ ]:
corr = df.select_dtypes('number').corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Корреляции числовых признаков')
plt.show()

### 5.6 Выводы EDA

| Что заметили | Что будем делать |
|---|---|
| Например: 30 % пропусков в столбце X | Заполнить медианой |
| … | … |

## 6. Предобработка

> **Подсказка.** Все преобразования, которые «обучаются» на данных (масштабирование, заполнение пропусков, кодирование), обучайте только на обучающей выборке, иначе произойдёт утечка данных. Для этого удобно использовать `Pipeline`.

### 6.1 План предобработки
Что делаем с пропусками, выбросами, категориями, масштабом. Для текста: векторизация (например, `TfidfVectorizer`) вместо кодирования столбцов.

### 6.2 Разделение на обучающую и тестовую выборки
Для классификации и регрессии. Для кластеризации пропустите этот пункт: вам нужен только набор признаков `X` и масштабирование.

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = [c for c in df.columns if c != TARGET]   # TODO: уберите лишние столбцы (id и т. п.)
X, y = df[FEATURES], df[TARGET]

# Для классификации добавьте stratify=y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

### 6.3 Конвейер предобработки

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = X_train.select_dtypes('number').columns
cat_cols = X_train.select_dtypes(exclude='number').columns

preprocess = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])

## 7. Базовая модель (baseline)

Базовая модель нужна, чтобы понимать, что значит «хороший результат»: любая настоящая модель должна быть заметно лучше неё. Для классификации это «всегда предсказывать самый частый класс», для регрессии «всегда предсказывать среднее». Для кластеризации в роли baseline можно взять K-Means с `k = 2–3`.

In [ ]:
from sklearn.dummy import DummyClassifier, DummyRegressor

# TODO: оставьте нужное — DummyClassifier(strategy='most_frequent') или DummyRegressor(strategy='mean')
baseline = Pipeline([('prep', preprocess),
                     ('model', DummyClassifier(strategy='most_frequent'))])
baseline.fit(X_train, y_train)
print('Baseline, результат на тесте:', baseline.score(X_test, y_test))

## 8. Основная модель

### 8.1 Какие модели выбрали и почему
Минимум две модели или две настройки одного метода (например, разные `k` или разная степень полинома).

In [ ]:
from sklearn.model_selection import cross_val_score

models = {
    'Модель 1': Pipeline([('prep', preprocess), ('model', ...)]),   # TODO
    'Модель 2': Pipeline([('prep', preprocess), ('model', ...)]),   # TODO
}

for name, m in models.items():
    scores = cross_val_score(m, X_train, y_train, cv=5, scoring='accuracy')  # TODO: метрика под вашу задачу
    print(f'{name}: {scores.mean():.3f} ± {scores.std():.3f}')

### 8.2 Подбор гиперпараметров

In [ ]:
from sklearn.model_selection import GridSearchCV

# TODO: пример для первой модели
# grid = GridSearchCV(models['Модель 1'],
#                     param_grid={'model__PARAM': [1, 3, 5, 7]},
#                     cv=5, scoring='accuracy')
# grid.fit(X_train, y_train)
# print(grid.best_params_, round(grid.best_score_, 3))

### 8.3 Только для кластеризации: выбор числа кластеров
Если ваш проект не про кластеризацию, удалите эту ячейку.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_scaled = StandardScaler().fit_transform(df.select_dtypes('number'))   # TODO: выберите нужные признаки

for k in range(2, 11):
    labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X_scaled)
    print(k, round(silhouette_score(X_scaled, labels), 3))

### 8.4 Сравнение моделей

| Модель | Параметры | Метрика (кросс-валидация) | Метрика (тест) | Комментарий |
|---|---|---|---|---|
| Baseline | — | … | … | … |
| Модель 1 | … | … | … | … |
| Модель 2 | … | … | … | … |

## 9. Оценка результатов

### 9.1 Метрики лучшей модели на тестовой выборке

In [ ]:
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                             mean_absolute_error, mean_squared_error, r2_score)

best = models['Модель 1']        # TODO: выберите лучшую
best.fit(X_train, y_train)
y_pred = best.predict(X_test)

# Классификация
print(classification_report(y_test, y_pred))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.show()

# Регрессия (раскомментируйте и уберите блок выше)
# print('MAE :', mean_absolute_error(y_test, y_pred))
# print('RMSE:', mean_squared_error(y_test, y_pred) ** 0.5)
# print('R2  :', r2_score(y_test, y_pred))
# plt.scatter(y_test, y_pred); plt.xlabel('Факт'); plt.ylabel('Прогноз'); plt.show()

# Multilabel: используйте hamming_loss и f1_score(average='micro' / 'macro')

### 9.2 Анализ ошибок и важность признаков
Где модель ошибается и почему (какие классы путает, на каких значениях большие ошибки). Если метод позволяет, покажите важность признаков или коэффициенты.

### 9.3 Только для кластеризации: профили кластеров
Опишите каждый кластер словами (например, «молодые клиенты с высокими тратами»).

In [ ]:
# df['cluster'] = labels                      # TODO: итоговые метки кластеров
# display(df.groupby('cluster').mean(numeric_only=True).round(2))
# df['cluster'].value_counts().sort_index().plot(kind='bar', title='Размеры кластеров')

## 10. Выводы

- **Что получилось:** ответ на цель проекта в 2–3 предложениях.
- **Какая модель лучше и почему:** со ссылкой на таблицу из 8.4.
- **Ограничения:** размер и качество данных, дисбаланс, возможная утечка, выбранные метрики.
- **Что можно улучшить:** какие шаги попробовали бы дальше.

## 11. Абзац для отчёта

> 5–8 предложений, которые потом без изменений идут в общий отчёт.
>
> Шаблон: «В проекте решалась задача … (тип). Использован датасет … (N объектов, M признаков). Применены методы … . Лучший результат показала модель …, метрика … составила … . Основной вывод: … . Ограничение: … .»

(Ваш абзац.)

## 12. Рефлексия

- **Что было самым сложным:**
- **Чему научился(лась):**
- **Какие вопросы остались:**

## 13. Чек-лист перед сдачей

- [ ] «Restart & Run All» выполняется без ошибок
- [ ] Удалены все блоки «Подсказка» и комментарии `TODO`
- [ ] Задан `random_state`
- [ ] Указаны ссылки на датасет и источники теории
- [ ] У графиков есть заголовки и подписи осей
- [ ] Выводы и теория написаны своими словами
- [ ] Файл назван по правилу `iterN_направление_тип_датасет.ipynb` и лежит в нужной папке репозитория
- [ ] Новые библиотеки добавлены в `requirements.txt`
- [ ] Создан pull request, назначен ревьюер

## 14. Источники

1. Название, ссылка
2. Название, ссылка